In [0]:
from pyspark.sql.functions import *

# Read bronze
df = spark.table("bronze.patients")

# Clean column names
for col_name in df.columns:
    df = df.withColumnRenamed(col_name, col_name.lower().replace(" ", "_"))

# Fix birth_date (SAFE parsing)
df = df.withColumn(
    "birth_date",
    coalesce(
        expr("try_to_date(birthdate_, 'yyyy-MM-dd')"),
        expr("try_to_date(birthdate_, 'dd-MM-yyyy')"),
        expr("try_to_date(birthdate_, 'MM-dd-yyyy')")
    )
)

# Fix death_date (SAFE parsing)
df = df.withColumn(
    "death_date",
    coalesce(
        expr("try_to_date(death_date, 'yyyy-MM-dd')"),
        expr("try_to_date(death_date, 'dd-MM-yyyy')"),
        expr("try_to_date(death_date, 'MM-dd-yyyy')")
    )
)

# Drop old column
df = df.drop("birthdate_")

# Trim strings
for c in df.columns:
    df = df.withColumn(c, trim(col(c)))

# Handle nulls
df = df.fillna({
    "gender": "UNKNOWN",
    "marital_status": "UNKNOWN"
})

# Remove bad rows
df = df.filter(col("id").isNotNull())


# Replace unknown + empty → NULL (for ALL columns)
df = df.replace(["", "unknown", "UNKNOWN", "Unknown"], None)

# Trim again (just to be safe)
for c in df.columns:
    df = df.withColumn(c, trim(col(c)))
    
# Clean names remove numbers
df = df.withColumn("first", regexp_replace(col("first"), "[^a-zA-Z]", ""))
df = df.withColumn("last", regexp_replace(col("last"), "[^a-zA-Z]", ""))
df = df.withColumn("maiden", regexp_replace(col("maiden"), "[^a-zA-Z]", ""))

# Remove system columns (like _line, _fivetran_synced)
df = df.select([
    col(c) for c in df.columns if not c.startswith("_")
])


# Save to silver
df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.patients")